In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from concurrent.futures import ThreadPoolExecutor
from contextlib import closing
from urllib.parse import quote_plus
import pandas as pd
import sqlite3
import threading
import random
import re
import time

In [ ]:
DATES = ("checkin=2026-11-10&checkout=2026-11-11&group_adults=2"
         "&no_rooms=1&group_children=0&selected_currency=EGP")
CITIES = ["Cairo", "Alexandria", "Dahab", "Hurghada", "Luxor"]
MAX_HOTELS_PER_CITY = 200
N_WORKERS = 3
HEADLESS = False
MAX_RETRIES = 2          # عدد المحاولات لكل فندق
DB_FILE = "booking.db"

db_lock = threading.Lock()

# Database functions

In [ ]:
def init_db():
    with closing(sqlite3.connect(DB_FILE)) as conn:
        conn.execute("PRAGMA journal_mode=WAL")
        conn.execute("""
            CREATE TABLE IF NOT EXISTS hotels (
                url TEXT PRIMARY KEY,
                city TEXT,
                name TEXT,
                min_price REAL,
                avg_price REAL,
                max_price REAL,
                lat TEXT,
                lon TEXT,
                facilities TEXT,
                categories_scores TEXT,
                pets TEXT
            )
        """)
        conn.commit()


In [ ]:
def save_hotel(data):
    with db_lock, closing(sqlite3.connect(DB_FILE, timeout=30)) as conn:
        conn.execute("""
            INSERT OR REPLACE INTO hotels
            (url, city, name, min_price, avg_price, max_price,
             lat, lon, facilities, categories_scores, pets)
            VALUES
            (:url, :city, :name, :min_price, :avg_price, :max_price,
             :lat, :lon, :facilities, :categories_scores, :pets)
        """, data)
        conn.commit()


In [ ]:
def get_done_urls(city):
    with closing(sqlite3.connect(DB_FILE)) as conn:
        cur = conn.execute("SELECT url FROM hotels WHERE city = ?", (city,))
        return {r[0] for r in cur.fetchall()}


In [ ]:
def count_hotels(city):
    with closing(sqlite3.connect(DB_FILE)) as conn:
        cur = conn.execute("SELECT COUNT(*) FROM hotels WHERE city = ?", (city,))
        return cur.fetchone()[0]


In [ ]:
def export_csv(output_file, city=None):
    with closing(sqlite3.connect(DB_FILE)) as conn:
        if city:
            df = pd.read_sql("SELECT * FROM hotels WHERE city = ?", conn, params=(city,))
        else:
            df = pd.read_sql("SELECT * FROM hotels", conn)
    df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"📄 Exported {len(df)} rows to {output_file}")
    return df

# Drive function

In [ ]:
def make_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--lang=en")
    options.add_argument("--window-size=1400,900")
    options.add_argument("--disable-notifications")
    if HEADLESS:
        options.add_argument("--headless=new")
    return webdriver.Chrome(options=options)


# Helper functions

In [ ]:
def close_popups(driver):
    for sel in [
        "button#onetrust-accept-btn-handler",
        "button[aria-label='Dismiss sign-in info.']",
    ]:
        try:
            WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
            ).click()
        except TimeoutException:
            pass


def get_text(el):
    t = (el.text or "").strip()
    if not t:
        t = (el.get_attribute("textContent") or "").strip()
    return t.replace("\xa0", " ")


def q(driver, selector, attr=None, multi=False):
    try:
        els = driver.find_elements(By.CSS_SELECTOR, selector)
        if not els:
            return [] if multi else None
        if multi:
            if attr:
                return [e.get_attribute(attr) for e in els if e.get_attribute(attr)]
            return [get_text(e) for e in els if get_text(e)]
        return els[0].get_attribute(attr) if attr else get_text(els[0])
    except Exception:
        return [] if multi else None


def get_pets_policy(all_texts):
    for txt in all_texts:
        lines = [l.strip() for l in txt.split("\n") if l.strip()]
        for i, line in enumerate(lines):
            if line.lower() == "pets" and i + 1 < len(lines):
                return lines[i + 1]
    return None


# Collect hotels data

In [ ]:
def close_popups(driver):
    for sel in [
        "button#onetrust-accept-btn-handler",
        "button[aria-label='Dismiss sign-in info.']",
    ]:
        try:
            WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, sel))
            ).click()
        except TimeoutException:
            pass


def get_text(el):
    t = (el.text or "").strip()
    if not t:
        t = (el.get_attribute("textContent") or "").strip()
    return t.replace("\xa0", " ")


def q(driver, selector, attr=None, multi=False):
    try:
        els = driver.find_elements(By.CSS_SELECTOR, selector)
        if not els:
            return [] if multi else None
        if multi:
            if attr:
                return [e.get_attribute(attr) for e in els if e.get_attribute(attr)]
            return [get_text(e) for e in els if get_text(e)]
        return els[0].get_attribute(attr) if attr else get_text(els[0])
    except Exception:
        return [] if multi else None


def get_pets_policy(all_texts):
    for txt in all_texts:
        lines = [l.strip() for l in txt.split("\n") if l.strip()]
        for i, line in enumerate(lines):
            if line.lower() == "pets" and i + 1 < len(lines):
                return lines[i + 1]
    return None


In [ ]:
def collect_hotel_links(driver, city, max_hotels):
    ss = quote_plus(f"{city}, Egypt")
    driver.get(f"https://www.booking.com/searchresults.en-us.html?ss={ss}&{DATES}")
    close_popups(driver)

    CARD_SEL = '[data-testid="property-card"] a[data-testid="title-link"]'
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, CARD_SEL))
        )
    except TimeoutException:
        driver.save_screenshot(f"debug_{city}.png")
        print(f"⚠️ {city}: مفيش كروت. العنوان: {driver.title!r}")
        print(f"   حفظت debug_{city}.png، افتحيها وشوفي فيه captcha ولا صفحة فاضية")
        return []

    links = {}
    stale_rounds = 0

    while len(links) < max_hotels and stale_rounds < 3:
        cards = driver.find_elements(By.CSS_SELECTOR, CARD_SEL)
        before = len(links)
        for a in cards:
            href = a.get_attribute("href")
            if href:
                links.setdefault(href.split("?")[0], href)

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)

        try:
            btn = driver.find_element(
                By.XPATH, "//button[.//span[contains(text(), 'Load more results')]]"
            )
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
        except Exception:
            pass

        stale_rounds = stale_rounds + 1 if len(links) == before else 0

    result = list(links.values())[:max_hotels]
    print(f"🔗 {city}: لقيت {len(result)} فندق")
    return result

# Scrapping one hotel

In [ ]:
def scrape_booking_hotel(driver, url, city):
    driver.get(url)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "h1.pp-header__title, h2.pp-header__title")
            )
        )
    except TimeoutException:
        raise TimeoutException(f"no title: {url}")

    driver.execute_script("window.scrollTo(0, 1500);")
    time.sleep(0.7)

    name = q(driver, "h1.pp-header__title") or q(driver, "h2.pp-header__title")
    latlng = q(driver, "#map_trigger_header", attr="data-atlas-latlng")
    facilities_raw = q(driver, '[data-testid="property-most-popular-facilities-wrapper"] li', multi=True)
    all_texts = q(driver, ".b99b6ef58f", multi=True)
    subscores = q(driver, '[data-testid="review-subscore"]', multi=True)
    price_texts = q(driver, "span.prco-valign-middle-helper", multi=True)


    if not all_texts:
        all_texts = [q(driver, "body") or ""]

    lat, lon = None, None
    if latlng:
        try:
            lat, lon = [x.strip() for x in latlng.split(",")]
        except Exception:
            pass

    facilities = list(dict.fromkeys(facilities_raw or []))

    categories_scores = {}
    for sub in subscores or []:
        parts = [p.strip() for p in sub.replace("\n", ",").split(",") if p.strip()]
        if len(parts) >= 2:
            categories_scores[parts[0]] = parts[1]

    prices = []
    for t in price_texts or []:
        m = re.search(r"EGP\s*([\d,]+(?:\.\d+)?)", t)
        if m:
            p = float(m.group(1).replace(",", ""))
            if p not in prices:
                prices.append(p)

    return {
        "city": city,
        "name": name,
        "min_price": min(prices) if prices else None,
        "avg_price": round(sum(prices) / len(prices), 2) if prices else None,
        "max_price": max(prices) if prices else None,
        "lat": lat,
        "lon": lon,
        "facilities": " | ".join(facilities),
        "categories_scores": str(categories_scores),
        "pets": get_pets_policy(all_texts),
        "url": url.split("?")[0],
    }

# Workers

In [ ]:
def run_chunk(chunk, city, worker_id):
    driver = make_driver()
    try:
        for i, link in enumerate(chunk, 1):
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    data = scrape_booking_hotel(driver, link, city)
                    save_hotel(data)
                    print(f"✅ [W{worker_id} {i}/{len(chunk)}] {data['name']} | "
                          f"{data['min_price']} - {data['max_price']}")
                    break
                except Exception as e:
                    if attempt == MAX_RETRIES:
                        print(f"❌ [W{worker_id}] فشل في {link}: {e}")
                    else:
                        time.sleep(2)
            time.sleep(random.uniform(0.5, 1.5))
    finally:
        driver.quit()


# scrapping whole city

In [ ]:
def scrape_city(city):
    start = time.time()
    init_db()

    main_driver = make_driver()
    try:
        links = collect_hotel_links(main_driver, city, MAX_HOTELS_PER_CITY)
    finally:
        main_driver.quit()

    # نتخطى اللي اتسكرب قبل كده
    done = get_done_urls(city)
    links = [l for l in links if l.split("?")[0] not in done]
    print(f"📍 {city}: باقي {len(links)} فندق (اتحفظ قبل كده {len(done)})")

    if links:
        chunks = [links[i::N_WORKERS] for i in range(N_WORKERS)]
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            futures = [
                ex.submit(run_chunk, chunk, city, idx + 1)
                for idx, chunk in enumerate(chunks) if chunk
            ]
            for f in futures:
                f.result()

    print(f"💾 {city}: إجمالي {count_hotels(city)} فندق في الداتا بيز")
    print(f"⏱️ time: {round((time.time() - start) / 60, 1)} minutes\n")


In [ ]:
for city in CITIES:
    scrape_city(city)

df_all = export_csv("all_hotels.csv")
print(df_all.groupby("city")[["min_price", "pets"]].apply(lambda g: g.isna().mean()))


In [ ]:
import sqlite3
import pyodbc
sqlite_conn = sqlite3.connect('booking.db')
sqlite_cursor = sqlite_conn.cursor()

table_name = 'hotels'


server_name = 'FIREFOX'
db_name = 'BookingData'

conn_str = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server_name};'
    f'DATABASE={db_name};'
    f'Trusted_connection=yes;'
)
sql_conn = pyodbc.connect(conn_str)
sql_cursor = sql_conn.cursor()


sqlite_cursor.execute(f"SELECT * FROM {table_name}")
rows = sqlite_cursor.fetchall()
columns = [description[0] for description in sqlite_cursor.description]
placeholders = ', '.join(['?'] * len(columns))
columns_str = ', '.join([f"[{col}]" for col in columns])

insert_query = f"INSERT INTO [{table_name}] ({columns_str}) VALUES ({placeholders})"

sql_cursor.executemany(insert_query, [tuple(row) for row in rows])
sql_conn.commit()

sqlite_conn.close()
sql_conn.close()
